# Experiment: Timm Image Classification CV

Objective:
- Train a stronger image classifier notebook that runs on Colab, Kaggle, or a local folder.
- Export fold metrics, OOF probabilities, test probabilities, and a submission file.

What changed vs the original script:
- Cross-platform path discovery for Colab, Kaggle, and local runs.
- Better augmentations and ImageNet normalization.
- Mixed precision, cosine LR schedule, gradient clipping, and early stopping.
- Safer submission logic using `LabelEncoder.inverse_transform(...)` instead of raw argmax integers.


In [ ]:
# Run once if timm / scikit-learn are missing.
import importlib.util
import subprocess
import sys

required_packages = {
    "timm": "timm",
    "scikit-learn": "sklearn",
}
missing_packages = [pkg for pkg, module_name in required_packages.items() if importlib.util.find_spec(module_name) is None]

if missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("All required packages are already installed.")


In [ ]:
from __future__ import annotations

import copy
import gc
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

import timm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

ImageFile.LOAD_TRUNCATED_IMAGES = True

CONFIG = {
    "seed": 42,
    "n_folds": 5,
    "epochs": 6,
    "batch_size": 16,
    "lr": 3e-5,
    "weight_decay": 1e-4,
    "img_size": 224,
    "model_name": "vit_base_patch16_224.augreg2_in21k_ft_in1k",
    "drop_path_rate": 0.10,
    "label_smoothing": 0.05,
    "grad_clip": 1.0,
    "early_stopping_patience": 2,
    "tta": 1,
    "num_workers": min(4, os.cpu_count() or 2),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
CONFIG["amp"] = CONFIG["device"] == "cuda"
CONFIG["pin_memory"] = CONFIG["device"] == "cuda"

# If you have more VRAM, try a larger backbone.
# CONFIG["model_name"] = "beitv2_large_patch16_224.in1k_ft_in22k_in1k"
# CONFIG["batch_size"] = 8

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CONFIG["seed"])

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device(CONFIG["device"])
OUTPUT_DIR = Path("notebook_outputs") / CONFIG["model_name"].replace("/", "_")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))
print("Torch:", torch.__version__)
print("Device:", DEVICE)


## Plan

- Start with a medium-size TIMM backbone that fits common Kaggle and Colab GPUs.
- Keep the pipeline reproducible with one config cell and one path-resolution cell.
- Save the same core artifacts every run: `oof_predictions.csv`, `test_predictions.csv`, `summary.json`, and `submission.csv`.


In [ ]:
# Set DATA_ROOT manually if auto-detection misses your dataset.
# Kaggle example: DATA_ROOT = "/kaggle/input/your-dataset-name"
# Colab example: DATA_ROOT = "/content"
DATA_ROOT = None

TRAIN_DIR_CANDIDATES = [
    ("train/train", "test/test"),
    ("train", "test"),
]

def resolve_data_paths(data_root: str | None = None) -> dict[str, Path]:
    search_roots: list[Path] = []
    if data_root:
        search_roots.append(Path(data_root))

    for base in [Path("/kaggle/input"), Path("/content"), Path(".")]:
        if base.exists():
            search_roots.append(base)

    checked_roots: list[str] = []
    for root in search_roots:
        csv_candidates = [root / "train.csv"]
        if root.is_dir():
            csv_candidates.extend(root.rglob("train.csv"))

        for train_csv in csv_candidates:
            parent = train_csv.parent
            sub_csv = parent / "sample_submission.csv"
            checked_roots.append(str(parent))
            if not sub_csv.exists():
                continue

            for train_rel, test_rel in TRAIN_DIR_CANDIDATES:
                train_dir = parent / train_rel
                test_dir = parent / test_rel
                if train_dir.exists() and test_dir.exists():
                    return {
                        "root": parent,
                        "train_csv": train_csv,
                        "sub_csv": sub_csv,
                        "train_dir": train_dir,
                        "test_dir": test_dir,
                    }

    checked_preview = "\n".join(f"- {path}" for path in checked_roots[:20])
    raise FileNotFoundError(
        "Could not auto-locate train.csv, sample_submission.csv, train dir, and test dir.\n"
        "Set DATA_ROOT manually. Checked these roots:\n"
        f"{checked_preview}"
    )

paths = resolve_data_paths(DATA_ROOT)
TRAIN_CSV = paths["train_csv"]
SUB_PATH = paths["sub_csv"]
TRAIN_DIR = paths["train_dir"]
TEST_DIR = paths["test_dir"]

print(json.dumps({key: str(value) for key, value in paths.items()}, indent=2))


In [ ]:
df = pd.read_csv(TRAIN_CSV)
sub = pd.read_csv(SUB_PATH)

def pick_first(columns: list[str], candidates: list[str], label: str) -> str:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise KeyError(f"Could not find {label}. Available columns: {columns}")

image_col = pick_first(df.columns.tolist(), ["image_name", "image_id", "id", "filename", "file_name"], "image column")
target_col = pick_first(df.columns.tolist(), ["class", "label", "target"], "target column")
test_id_col = pick_first(sub.columns.tolist(), ["id", "image_name", "image_id", "filename", "file_name"], "test image column")
answer_col = "answer" if "answer" in sub.columns else sub.columns[-1]

df = df.copy()
sub = sub.copy()
le = LabelEncoder()
df["label"] = le.fit_transform(df[target_col])
num_classes = len(le.classes_)

def build_stem_map(directory: Path) -> dict[str, Path]:
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return {
        path.stem: path
        for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in valid_exts
    }

TRAIN_FILE_MAP = build_stem_map(TRAIN_DIR)
TEST_FILE_MAP = build_stem_map(TEST_DIR)

print(f"train rows: {len(df):,}")
print(f"test rows: {len(sub):,}")
print(f"num_classes: {num_classes}")
print(f"train image files indexed: {len(TRAIN_FILE_MAP):,}")
print(f"test image files indexed: {len(TEST_FILE_MAP):,}")
display(df.head())
display(sub.head())


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tfm = T.Compose([
    T.RandomResizedCrop(
        CONFIG["img_size"],
        scale=(0.80, 1.0),
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    ),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)], p=0.5),
    T.RandomRotation(12),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.15, scale=(0.02, 0.12), ratio=(0.3, 3.3), value="random"),
])

val_tfm = T.Compose([
    T.Resize(int(CONFIG["img_size"] * 1.14), interpolation=InterpolationMode.BICUBIC, antialias=True),
    T.CenterCrop(CONFIG["img_size"]),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def resolve_image_path(image_ref: str, base_dir: Path, file_map: dict[str, Path]) -> Path:
    image_path = Path(image_ref)
    if image_path.is_absolute() and image_path.exists():
        return image_path

    candidate = base_dir / image_ref
    if candidate.exists():
        return candidate

    stem_match = file_map.get(Path(image_ref).stem)
    if stem_match is not None:
        return stem_match

    raise FileNotFoundError(f"Could not resolve image: {image_ref}")

class ImageDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        image_dir: Path,
        file_map: dict[str, Path],
        image_column: str,
        transform: T.Compose,
        label_column: str | None = None,
    ) -> None:
        self.frame = frame.reset_index(drop=True)
        self.image_dir = image_dir
        self.file_map = file_map
        self.image_column = image_column
        self.transform = transform
        self.label_column = label_column

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        image_path = resolve_image_path(str(row[self.image_column]), self.image_dir, self.file_map)
        image = Image.open(image_path).convert("RGB")
        tensor = self.transform(image)

        if self.label_column is None:
            return tensor

        label = int(row[self.label_column])
        return tensor, label

def make_loaders(train_frame: pd.DataFrame, val_frame: pd.DataFrame):
    train_ds = ImageDataset(train_frame, TRAIN_DIR, TRAIN_FILE_MAP, image_col, train_tfm, label_column="label")
    val_ds = ImageDataset(val_frame, TRAIN_DIR, TRAIN_FILE_MAP, image_col, val_tfm, label_column="label")
    test_ds = ImageDataset(sub, TEST_DIR, TEST_FILE_MAP, test_id_col, val_tfm, label_column=None)

    loader_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "num_workers": CONFIG["num_workers"],
        "pin_memory": CONFIG["pin_memory"],
    }

    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader


In [ ]:
def build_model(num_classes: int) -> nn.Module:
    model = timm.create_model(
        CONFIG["model_name"],
        pretrained=True,
        num_classes=num_classes,
        drop_path_rate=CONFIG["drop_path_rate"],
    )
    return model.to(DEVICE)

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
) -> float:
    model.train()
    running_loss = 0.0

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CONFIG["amp"]):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

@torch.no_grad()
def validate(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> dict[str, np.ndarray | float]:
    model.eval()
    running_loss = 0.0
    all_probs = []
    all_targets = []

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with autocast(enabled=CONFIG["amp"]):
            logits = model(images)
            loss = criterion(logits, targets)

        running_loss += loss.item() * images.size(0)
        all_probs.append(logits.softmax(dim=1).cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    preds = probs.argmax(axis=1)
    return {
        "loss": running_loss / len(loader.dataset),
        "acc": float(accuracy_score(targets, preds)),
        "probs": probs,
        "targets": targets,
    }

@torch.no_grad()
def predict_probs(model: nn.Module, loader: DataLoader, tta: int = 1) -> np.ndarray:
    model.eval()
    outputs = []

    for batch in loader:
        images = batch[0] if isinstance(batch, (tuple, list)) else batch
        images = images.to(DEVICE, non_blocking=True)
        probs = torch.zeros((images.size(0), num_classes), device=DEVICE)

        for tta_idx in range(tta):
            batch_images = torch.flip(images, dims=[3]) if tta_idx == 1 else images
            with autocast(enabled=CONFIG["amp"]):
                logits = model(batch_images)
            probs += logits.softmax(dim=1)

        probs /= tta
        outputs.append(probs.cpu().numpy())

    return np.concatenate(outputs, axis=0)


In [ ]:
skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])

oof_probs = np.zeros((len(df), num_classes), dtype=np.float32)
test_probs = np.zeros((len(sub), num_classes), dtype=np.float32)
fold_scores: list[float] = []
histories: list[dict[str, object]] = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label"]), start=1):
    print(f"\n========== FOLD {fold}/{CONFIG['n_folds']} ==========")
    train_frame = df.iloc[train_idx].reset_index(drop=True)
    val_frame = df.iloc[val_idx].reset_index(drop=True)
    train_loader, val_loader, test_loader = make_loaders(train_frame, val_frame)

    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
    scaler = GradScaler(enabled=CONFIG["amp"])

    best_acc = -1.0
    best_state = None
    patience = 0
    fold_history = []

    for epoch in range(1, CONFIG["epochs"] + 1):
        started_at = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_metrics = validate(model, val_loader, criterion)
        scheduler.step()
        elapsed = time.time() - started_at

        row = {
            "fold": fold,
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_metrics["loss"]),
            "val_acc": float(val_metrics["acc"]),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "seconds": round(elapsed, 2),
        }
        fold_history.append(row)

        print(
            f"Fold {fold} | Epoch {epoch:02d}/{CONFIG['epochs']} | "
            f"train_loss={row['train_loss']:.4f} | val_loss={row['val_loss']:.4f} | "
            f"val_acc={row['val_acc']:.4f} | lr={row['lr']:.2e} | time={row['seconds']:.1f}s"
        )

        if row["val_acc"] > best_acc:
            best_acc = row["val_acc"]
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= CONFIG["early_stopping_patience"]:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_state)
    val_metrics = validate(model, val_loader, criterion)
    oof_probs[val_idx] = val_metrics["probs"]
    fold_test_probs = predict_probs(model, test_loader, tta=CONFIG["tta"])
    test_probs += fold_test_probs / CONFIG["n_folds"]
    fold_scores.append(float(val_metrics["acc"]))
    histories.append({"fold": fold, "best_acc": float(val_metrics["acc"]), "history": fold_history})

    torch.save(best_state, OUTPUT_DIR / f"fold_{fold}.pth")

    del model, optimizer, scheduler, criterion, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
oof_pred_idx = oof_probs.argmax(axis=1)
cv_acc = float(accuracy_score(df["label"].values, oof_pred_idx))

summary = {
    "model_name": CONFIG["model_name"],
    "img_size": CONFIG["img_size"],
    "fold_scores": [round(score, 6) for score in fold_scores],
    "mean_fold_acc": float(np.mean(fold_scores)),
    "oof_cv_acc": cv_acc,
    "label_mapping": {int(i): str(label) for i, label in enumerate(le.classes_)},
    "config": CONFIG,
}

class_columns = [f"prob_class_{i}" for i in range(num_classes)]

oof_export = pd.DataFrame(oof_probs, columns=class_columns)
oof_export.insert(0, "row_id", np.arange(len(df)))
oof_export.insert(1, "true_label", df[target_col].astype(str).values)
oof_export.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)

test_export = pd.DataFrame(test_probs, columns=class_columns)
test_export.insert(0, test_id_col, sub[test_id_col].astype(str).values)
test_export.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

submission = sub.copy()
submission[answer_col] = le.inverse_transform(test_probs.argmax(axis=1))
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)

with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Fold scores:", [round(score, 4) for score in fold_scores])
print("Mean fold acc:", round(np.mean(fold_scores), 4))
print("OOF CV acc:", round(cv_acc, 4))
print(f"Saved artifacts to: {OUTPUT_DIR.resolve()}")
display(pd.DataFrame(histories[-1]["history"]))
display(submission.head())


## Next steps

- If this runs end-to-end cleanly, increase `epochs` first before jumping to a heavier backbone.
- If your GPU has enough VRAM, try BEiTv2 or EVA02 and reduce `batch_size` accordingly.
- If leaderboard predictions prefer slightly more stable inference, set `CONFIG["tta"] = 2` for horizontal-flip TTA.
